# UCS420: Cognitive Computing — Lab Assignment 9 (Part A)
## Cognitive Assistant for Reading Medicine Bottle Labels
*Vision-based Cognitive System (Landing AI theme)*

The assistant **sees** a medicine label (photo), **reads** it (OCR), **understands** what each line means (a trained line-type classifier plus entity extraction), **reasons** about it (drug knowledge base plus safety rules), and **responds** in text, speech and answers to follow-up questions. This is useful, for example, for elderly or visually-impaired users.

```
 ┌──────────┐   ┌──────────────┐   ┌────────────────────┐   ┌─────────────────┐   ┌──────────────┐
 │  Image   │──▶│  Perception  │──▶│   Comprehension    │──▶│    Reasoning    │──▶│   Response   │
 │ (bottle) │   │ preprocess + │   │ line classifier +  │   │ drug KB + rules │   │ summary, Q&A │
 └──────────┘   │ EasyOCR      │   │ regex/fuzzy NER    │   │ (expiry, dose…) │   │ text-to-speech│
                └──────────────┘   └────────────────────┘   └─────────────────┘   └──────────────┘
```

**Datasets**
1. **Synthetic labelled medicine labels**: generated in §3 with known ground truth (drug, strength, expiry…), so the full pipeline can be scored quantitatively.
2. **Real medicine packaging photos**: [`mennox/Italian-OTC-medicines`](https://huggingface.co/datasets/mennox/Italian-OTC-medicines) on Hugging Face (718 images, Apache-2.0), downloaded from the internet in §9. You can also upload your own bottle photos.

*Runtime → Change runtime type → **T4 GPU*** (optional; makes EasyOCR faster). Then **Runtime → Run all**.

## 0. Setup

In [ ]:
!pip -q install easyocr gTTS

In [ ]:
import re, os, json, random, difflib, warnings
from datetime import date, datetime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

random.seed(42); np.random.seed(42)
TODAY = date.today()

# OCR engine: EasyOCR (deep-learning CRAFT text detector + CRNN recogniser); Tesseract as fallback
try:
    import easyocr, torch
    reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
    OCR_ENGINE = "easyocr"
except Exception as e:
    import pytesseract
    OCR_ENGINE = "tesseract"
print("OCR engine:", OCR_ENGINE, "| Today:", TODAY)

## 1. Domain Knowledge Base (Drugs)
A small knowledge base of common medicines sold in India. Each entry has generic name, brand names, drug class, uses, key warnings, common side effects, whether it is prescription-only (Schedule H) and the usual **maximum daily dose for adults**, which the dose-check rule uses.

*General label-level information only; not medical advice.*

In [ ]:
DRUG_KB = {
    "paracetamol": {"brands": ["dolo", "crocin", "calpol", "tylenol", "acetaminophen", "paracip"],
        "class": "Analgesic / antipyretic", "rx": False, "max_daily_mg": 4000,
        "uses": "relief of fever and mild-to-moderate pain (headache, body ache, toothache)",
        "warnings": "do not exceed the maximum daily dose; avoid taking with other paracetamol-containing products; take extra care with liver disease or regular alcohol use",
        "side_effects": "rare when used correctly; overdose can cause serious liver damage"},
    "ibuprofen": {"brands": ["brufen", "advil", "ibugesic", "combiflam"],
        "class": "NSAID (anti-inflammatory painkiller)", "rx": False, "max_daily_mg": 1200,
        "uses": "pain, inflammation and fever (muscle pain, period pain, dental pain)",
        "warnings": "take with food; avoid with stomach ulcers, kidney disease, heart failure or in late pregnancy; can raise blood pressure",
        "side_effects": "indigestion, nausea, heartburn; rarely stomach bleeding"},
    "aspirin": {"brands": ["ecosprin", "disprin", "loprin"],
        "class": "Antiplatelet / NSAID", "rx": False, "max_daily_mg": 4000,
        "uses": "pain and fever at higher doses; low dose (75–150 mg) to prevent blood clots on a doctor's advice",
        "warnings": "not for children under 16; avoid with bleeding disorders or stomach ulcers; increases bleeding risk",
        "side_effects": "stomach irritation, easy bruising or bleeding"},
    "cetirizine": {"brands": ["cetzine", "zyrtec", "alerid", "okacet"],
        "class": "Antihistamine", "rx": False, "max_daily_mg": 10,
        "uses": "allergies — sneezing, runny nose, itchy eyes, hives",
        "warnings": "may cause drowsiness; avoid alcohol and take care when driving",
        "side_effects": "drowsiness, dry mouth, headache"},
    "amoxicillin": {"brands": ["mox", "novamox", "amoxil", "augmentin"],
        "class": "Penicillin antibiotic", "rx": True, "max_daily_mg": 3000,
        "uses": "bacterial infections (ear, throat, chest, urinary tract)",
        "warnings": "do not take if allergic to penicillin; complete the full course even if you feel better",
        "side_effects": "diarrhoea, nausea, skin rash"},
    "azithromycin": {"brands": ["azithral", "azee", "zithromax"],
        "class": "Macrolide antibiotic", "rx": True, "max_daily_mg": 500,
        "uses": "bacterial infections of the chest, throat, skin and ears",
        "warnings": "complete the course; tell your doctor about heart-rhythm problems or liver disease",
        "side_effects": "nausea, diarrhoea, stomach pain"},
    "metformin": {"brands": ["glycomet", "glucophage", "obimet"],
        "class": "Antidiabetic (biguanide)", "rx": True, "max_daily_mg": 2550,
        "uses": "type 2 diabetes — lowers blood sugar",
        "warnings": "take with meals; avoid excess alcohol; may need to be stopped before scans with contrast dye",
        "side_effects": "nausea, diarrhoea, metallic taste"},
    "pantoprazole": {"brands": ["pan", "pantocid", "protonix"],
        "class": "Proton-pump inhibitor (antacid)", "rx": True, "max_daily_mg": 80,
        "uses": "acidity, heartburn, stomach ulcers, GERD",
        "warnings": "take before food (usually 30–60 min before breakfast); long-term use only on medical advice",
        "side_effects": "headache, diarrhoea, stomach pain"},
    "omeprazole": {"brands": ["omez", "prilosec"],
        "class": "Proton-pump inhibitor (antacid)", "rx": True, "max_daily_mg": 40,
        "uses": "acidity, heartburn, stomach ulcers, GERD",
        "warnings": "take before meals; long-term use only on medical advice",
        "side_effects": "headache, nausea, diarrhoea"},
    "atorvastatin": {"brands": ["atorva", "lipitor", "storvas"],
        "class": "Statin (cholesterol-lowering)", "rx": True, "max_daily_mg": 80,
        "uses": "lowering LDL cholesterol and reducing heart attack / stroke risk",
        "warnings": "report unexplained muscle pain; avoid large amounts of grapefruit juice",
        "side_effects": "muscle aches, headache, digestive upset"},
    "amlodipine": {"brands": ["amlong", "norvasc", "stamlo"],
        "class": "Calcium-channel blocker (BP)", "rx": True, "max_daily_mg": 10,
        "uses": "high blood pressure and angina",
        "warnings": "do not stop suddenly; rise slowly from sitting to avoid dizziness",
        "side_effects": "ankle swelling, flushing, headache"},
    "montelukast": {"brands": ["montair", "singulair", "romilast"],
        "class": "Leukotriene receptor antagonist", "rx": True, "max_daily_mg": 10,
        "uses": "asthma prevention and allergic rhinitis",
        "warnings": "not for a sudden asthma attack; report mood or sleep changes",
        "side_effects": "headache, stomach pain, sleep disturbance"},
}

# Allergy classes — used by the allergy-conflict rule
ALLERGY_GROUPS = {"penicillin": ["amoxicillin"], "nsaid": ["ibuprofen", "aspirin"],
                  "sulfa": [], "aspirin": ["aspirin"]}

# Look-up table of every name (generic + brands) → generic
NAME_INDEX = {g: g for g in DRUG_KB}
for g, info in DRUG_KB.items():
    for b in info["brands"]:
        NAME_INDEX[b] = g
print(f"{len(DRUG_KB)} drugs, {len(NAME_INDEX)} searchable names")

## 2. Label Line Taxonomy
Every text line on a medicine label falls into one of these **9 field types**. A classifier is trained (§4) to recognise them from (noisy) OCR text.

In [ ]:
FIELDS = {
    "drug_name":    "Brand / generic name, usually with strength (e.g. 'DOLO 650 — Paracetamol Tablets IP 650 mg')",
    "composition":  "'Each tablet contains: …'",
    "directions":   "How to take it (dose, frequency, with/without food)",
    "warning":      "Cautions, contraindications, 'keep out of reach of children'",
    "storage":      "'Store below 25°C in a dry place…'",
    "expiry":       "Expiry / manufacturing dates",
    "batch":        "Batch / lot number, Mfg. Lic. No.",
    "manufacturer": "'Manufactured by / Marketed by …' and address",
    "rx":           "Prescription notice — 'Rx', 'Schedule H drug'",
}
pd.DataFrame(FIELDS.items(), columns=["Field", "Description"])

## 3. Synthetic Medicine Label Dataset
Real labels don't come with ground truth, so we **generate realistic labels with known content**: random drug, strength, directions, warnings, storage, batch, manufacturing and expiry dates (about 30% already expired). They are rendered as images with camera-like distortions (rotation, blur, noise, lighting).
This gives (a) thousands of labelled **text lines** to train the line classifier and (b) labelled **images** to evaluate the full vision pipeline end-to-end.

In [ ]:
STRENGTHS = {"paracetamol": [500, 650], "ibuprofen": [200, 400], "aspirin": [75, 150, 325],
             "cetirizine": [10], "amoxicillin": [250, 500], "azithromycin": [250, 500],
             "metformin": [500, 850, 1000], "pantoprazole": [20, 40], "omeprazole": [20],
             "atorvastatin": [10, 20, 40], "amlodipine": [5, 10], "montelukast": [10]}
FORMS = ["Tablets", "Capsules"]
MANUFACTURERS = ["Sunrise Pharma Ltd.", "Medilife Healthcare Pvt. Ltd.", "Novagen Laboratories",
                 "Arogya Remedies Ltd.", "Himgiri Drugs & Chemicals", "Zenova Biotech Pvt. Ltd."]
CITIES = ["Baddi, H.P.", "Ahmedabad, Gujarat", "Sikkim", "Haridwar, Uttarakhand", "Hyderabad, Telangana", "Goa"]
MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]

DIRECTIONS = [
    "Dosage: Take {n} {unit} {freq} or as directed by the physician.",
    "Directions: {n} {unit} {freq} after meals.",
    "Adults: {n} {unit} {freq}. Swallow whole with water.",
    "Take {n} {unit} {freq} with food.",
    "Dose: {n} {unit} {freq}, do not chew or crush.",
]
FREQS = [("once daily", 1), ("twice daily", 2), ("three times a day", 3), ("every 8 hours", 3),
         ("every 6 hours", 4), ("once a day", 1), ("2 times a day", 2), ("thrice daily", 3)]
WARNINGS = [
    "Warning: Do not exceed the recommended dose.",
    "Keep out of reach of children.",
    "Caution: Not to be sold without prescription.",
    "Do not use if seal is broken.",
    "Warning: May cause drowsiness. Avoid alcohol.",
    "Consult your doctor if pregnant or breastfeeding.",
    "Discontinue use if rash or allergic reaction occurs.",
]
STORAGE = ["Store below 25°C in a dry place.", "Store in a cool, dry place away from direct sunlight.",
           "Protect from light and moisture.", "Store below 30°C. Protect from moisture."]
RX_LINES = ["Rx only", "SCHEDULE H PRESCRIPTION DRUG - CAUTION", "Rx Prescription Drug"]

def _fmt_date(d, style):
    if style == 0: return f"{d.month:02d}/{d.year}"
    if style == 1: return f"{MONTHS[d.month-1]}. {d.year}"
    if style == 2: return f"{MONTHS[d.month-1]} {d.year}"
    return f"{d.month:02d}-{d.year}"

def make_label_record():
    g = random.choice(list(DRUG_KB))
    brand = random.choice(DRUG_KB[g]["brands"]).upper()
    s = random.choice(STRENGTHS[g]); form = random.choice(FORMS)
    unit = "tablet" if form == "Tablets" else "capsule"
    # realistic directions: never above the drug's maximum daily dose
    options = [(f, k, n) for f, k in FREQS for n in (1, 2) if n * k * s <= DRUG_KB[g]["max_daily_mg"]]
    freq, per_day, n = random.choice(options)
    unit_txt = unit + ("s" if n > 1 else "")
    mfg = date(random.randint(2023, 2026), random.randint(1, 12), 1)
    shelf = random.choice([12, 18, 24, 36])
    exp_m = mfg.month - 1 + shelf
    exp = date(mfg.year + exp_m // 12, exp_m % 12 + 1, 1)
    ds = random.randint(0, 3)
    rec = {
        "generic": g, "brand": brand, "strength_mg": s, "form": form,
        "dose_units": n, "per_day": per_day, "mfg": mfg.isoformat(), "exp": exp.isoformat(),
        "batch": f"{random.choice('ABCDEFGHKMNPRT')}{random.choice('ABCDEFGHKMNPRT')}{random.randint(1000, 99999)}",
        "rx": DRUG_KB[g]["rx"],
    }
    lines = [
        ("drug_name", f"{brand} {s}"),
        ("drug_name", f"{g.capitalize()} {form} IP {s} mg"),
        ("composition", f"Each {unit} contains: {g.capitalize()} IP {s} mg"),
        ("directions", random.choice(DIRECTIONS).format(n=n, unit=unit_txt, freq=freq)),
        ("warning", random.choice(WARNINGS)),
        ("storage", random.choice(STORAGE)),
        ("batch", f"{random.choice(['Batch No.', 'B.No.', 'Lot No.', 'Batch'])} {rec['batch']}"),
        ("expiry", f"MFG. {_fmt_date(mfg, ds)}"),
        ("expiry", f"{random.choice(['EXP.', 'Exp. Date', 'Expiry', 'Use before'])} {_fmt_date(exp, ds)}"),
        ("manufacturer", f"{random.choice(['Manufactured by:', 'Mfd. by:', 'Marketed by:'])} {random.choice(MANUFACTURERS)}"),
        ("manufacturer", f"{random.choice(['Plot', 'Unit', 'Khasra No.'])} {random.randint(2, 250)}, {random.choice(CITIES)}"),
    ]
    if rec["rx"]:
        lines.insert(0, ("rx", random.choice(RX_LINES)))
    if random.random() < 0.5:
        lines.insert(6, ("warning", random.choice(WARNINGS)))
    rec["lines"] = lines
    return rec

FONT_PATHS = ["/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
BOLD_PATHS = ["/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"]

def _font(paths, size):
    for p in paths:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default(size=size)

def render_label(rec, augment=True, hard=False):
    W, H = 1000, 80 + 46 * len(rec["lines"])
    bg = random.choice([(255, 255, 255), (250, 247, 235), (235, 245, 255), (245, 255, 245)])
    band = random.choice([(200, 30, 45), (20, 90, 170), (0, 130, 90), (120, 40, 140)])
    img = Image.new("RGB", (W, H), bg)
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, W, 18], fill=band); d.rectangle([0, H - 18, W, H], fill=band)
    y = 34
    for field, text in rec["lines"]:
        if field == "drug_name" and text.startswith(rec["brand"]):
            d.text((40, y), text, fill=band, font=_font(BOLD_PATHS, 40)); y += 54
        else:
            d.text((40, y), text, fill=(25, 25, 25), font=_font(FONT_PATHS, 26)); y += 44
    img = img.crop((0, 0, W, min(H, y + 30)))
    if augment:  # simulate a phone photo of a bottle
        rot, blur, noise, light = ((7, (1.0, 2.0), (8, 18), (0.5, 1.2)) if hard else
                                   (4, (0.3, 1.2), (2, 9), (0.75, 1.15)))
        img = img.rotate(random.uniform(-rot, rot), expand=True, fillcolor=(200, 200, 200))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(*light))
        if hard:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.5, 0.8))
            w, h = img.size
            img = img.resize((int(w * 0.45), int(h * 0.45))).resize((w, h))   # low-resolution camera
        if hard or random.random() < 0.6:
            img = img.filter(ImageFilter.GaussianBlur(random.uniform(*blur)))
        arr = np.array(img).astype(np.int16)
        arr += np.random.normal(0, random.uniform(*noise), arr.shape).astype(np.int16)
        img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
        if hard:  # JPEG compression artefacts
            import io
            buf = io.BytesIO(); img.save(buf, "JPEG", quality=random.randint(20, 40)); img = Image.open(buf).convert("RGB")
    return img

records = [make_label_record() for _ in range(600)]
expired_share = np.mean([date.fromisoformat(r["exp"]) < TODAY for r in records])
print(f"Generated {len(records)} label records | {expired_share:.0%} already expired")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, r in zip(axes, records[:3]):
    ax.imshow(render_label(r)); ax.axis("off"); ax.set_title(f"{r['brand']} ({r['generic']}), exp {r['exp'][:7]}")
plt.tight_layout(); plt.show()

## 4. Model: Label-Line Field Classifier
**Input:** one text line (from OCR). **Output:** its field type.
**Features:** TF-IDF of character n-grams (2–5), which is robust to OCR spelling errors like `Expiny` or `0ATCH`. **Classifier:** Logistic Regression.
To make the model robust to real OCR, the training lines get **simulated OCR noise** (character swaps such as O↔0, l↔1, S↔5, and dropped or doubled characters).

In [ ]:
OCR_CONFUSIONS = {"o": "0", "0": "o", "l": "1", "1": "l", "i": "l", "s": "5", "5": "s", "e": "c",
                  "b": "8", "8": "b", "g": "9", "rn": "m", "m": "rn"}

def ocr_noise(text, p=0.06):
    out, i = [], 0
    t = text.lower()
    while i < len(t):
        if random.random() < p:
            op = random.random()
            two = t[i:i+2]
            if two in OCR_CONFUSIONS and op < 0.3:
                out.append(OCR_CONFUSIONS[two]); i += 2; continue
            if t[i] in OCR_CONFUSIONS and op < 0.7:
                out.append(OCR_CONFUSIONS[t[i]])
            elif op < 0.85:
                pass                                # drop char
            else:
                out.append(t[i] * 2)                # double char
        else:
            out.append(t[i])
        i += 1
    return "".join(out)

line_rows = []
for r in records:
    for field, text in r["lines"]:
        line_rows.append((text.lower(), field))
        line_rows.append((ocr_noise(text), field))
lines_df = pd.DataFrame(line_rows, columns=["text", "field"]).drop_duplicates()
print(f"{len(lines_df)} labelled lines\n", lines_df["field"].value_counts().to_string())

Xtr, Xte, ytr, yte = train_test_split(lines_df["text"], lines_df["field"], test_size=0.2,
                                      stratify=lines_df["field"], random_state=42)
line_clf = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)),
    ("lr", LogisticRegression(C=5, max_iter=3000)),
]).fit(Xtr, ytr)
pred = line_clf.predict(Xte)
print(f"\nLine-classifier test accuracy (clean held-out lines): {accuracy_score(yte, pred):.3f}")
print("Near-perfect here because synthetic lines share templates; §9 measures it on real OCR output.\n")
print(classification_report(yte, pred, digits=3))

labels = sorted(lines_df["field"].unique())
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(yte, pred, labels=labels), annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Line field classifier — confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()

## 5. Perception: Image Pre-processing + OCR
Pre-processing: upscale small images → grayscale → denoise → CLAHE contrast enhancement → **deskew** (projection-profile search for the rotation angle that makes text rows sharpest). OCR output is grouped into **lines** (by vertical position) with a confidence score per line.

In [ ]:
def preprocess_image(img):
    """PIL/np image → enhanced grayscale np.uint8 array for OCR."""
    arr = np.array(img.convert("RGB")) if isinstance(img, Image.Image) else img
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY) if arr.ndim == 3 else arr
    h, w = gray.shape
    if max(h, w) < 1200:
        f = 1200 / max(h, w)
        gray = cv2.resize(gray, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    gray = cv2.fastNlMeansDenoising(gray, h=7)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return deskew(gray)

def deskew(gray, max_angle=8, step=0.5):
    """Find the rotation that maximises the variance of row sums of the text mask, then rotate."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5)
    mask = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    h, w = mask.shape
    best_angle, best_score = 0.0, -1
    for a in np.arange(-max_angle, max_angle + step, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        rot = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST)
        score = np.var(rot.sum(axis=1))
        if score > best_score:
            best_angle, best_score = a, score
    if abs(best_angle) < 0.25:
        return gray
    H, W = gray.shape
    M = cv2.getRotationMatrix2D((W / 2, H / 2), best_angle, 1.0)
    return cv2.warpAffine(gray, M, (W, H), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def _group_lines(boxes):
    """boxes: list of (x, y_center, height, text, conf) → list of (line_text, mean_conf)."""
    boxes = sorted(boxes, key=lambda b: b[1])
    lines, cur = [], []
    for b in boxes:
        if cur and abs(b[1] - np.mean([c[1] for c in cur])) > 0.6 * np.mean([c[2] for c in cur]):
            lines.append(cur); cur = []
        cur.append(b)
    if cur: lines.append(cur)
    out = []
    for ln in lines:
        ln = sorted(ln, key=lambda b: b[0])
        out.append((" ".join(b[3] for b in ln).strip(), float(np.mean([b[4] for b in ln]))))
    return [l for l in out if l[0]]

def run_ocr(img):
    gray = preprocess_image(img)
    boxes = []
    if OCR_ENGINE == "easyocr":
        for bbox, text, conf in reader.readtext(gray):
            ys = [p[1] for p in bbox]; xs = [p[0] for p in bbox]
            boxes.append((min(xs), (min(ys) + max(ys)) / 2, max(ys) - min(ys), text, conf))
    else:
        d = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT, config="--psm 6")
        groups = {}
        for i, t in enumerate(d["text"]):
            if t.strip() and float(d["conf"][i]) >= 0:
                key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
                groups.setdefault(key, []).append((d["left"][i], t, float(d["conf"][i]) / 100))
        return [(" ".join(w[1] for w in sorted(g)), float(np.mean([w[2] for w in g]))) for g in groups.values()]
    return _group_lines(boxes)

sample = render_label(records[0])
for text, conf in run_ocr(sample):
    print(f"{conf:.2f}  {text}")

## 6. Comprehension: Information Extraction
- **Line field:** from the trained classifier (§4)
- **Drug identity:** fuzzy matching (`difflib`) of every word against all generic and brand names, tolerant to OCR errors
- **Strength, dates, batch, dose and frequency:** regular expressions

In [ ]:
MONTH_MAP = {m: i + 1 for i, m in enumerate(MONTHS)}
NUM_WORDS = {"one": 1, "two": 2, "three": 3, "half": 0.5}

def _parse_date(s):
    s = s.upper().replace("O", "0") if re.search(r"\d", s) else s.upper()
    m = re.search(r"\b(0?[1-9]|1[0-2])\s*[/\-.]\s*((?:20)?\d{2})\b", s)
    if m:
        mo, yr = int(m.group(1)), int(m.group(2)); yr += 2000 if yr < 100 else 0
        return date(yr, mo, 1)
    m = re.search(r"\b(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*\.?\s*[,\-/]?\s*((?:20)?\d{2})\b", s.replace("0CT", "OCT"))
    if m:
        yr = int(m.group(2)); yr += 2000 if yr < 100 else 0
        return date(yr, MONTH_MAP[m.group(1)], 1)
    return None

def identify_drug(text):
    words = re.findall(r"[a-z]{3,}", text.lower())
    best, score = None, 0.0
    for w in words:
        for name, generic in NAME_INDEX.items():
            if len(name) < 3: continue
            r = difflib.SequenceMatcher(None, w, name).ratio()
            if len(name) <= 4 and r < 1.0: continue          # short brand names must match exactly
            if r > score:
                best, score = generic, r
    return (best, round(score, 2)) if score >= 0.8 else (None, round(score, 2))

def parse_daily_units(directions):
    t = directions.lower()
    n = re.search(r"\b(\d+|one|two|three|half)\s*(tablet|tab|capsule|cap)", t)
    units = (float(n.group(1)) if n.group(1).isdigit() else NUM_WORDS[n.group(1)]) if n else None
    freq = None
    if re.search(r"once (a |per )?da(il)?y|once daily|\bod\b", t): freq = 1
    elif re.search(r"twice|\bbd\b|2 times", t): freq = 2
    elif re.search(r"thrice|three times|3 times|\btds\b", t): freq = 3
    elif re.search(r"four times|4 times|\bqid\b", t): freq = 4
    h = re.search(r"every (\d+) ?h", t)
    if h: freq = 24 // int(h.group(1))
    return units, freq

def extract_info(ocr_lines):
    info = {"lines": [], "drug": None, "drug_match": 0.0, "strength_mg": None, "mfg": None,
            "exp": None, "batch": None, "directions": [], "warnings": [], "storage": [],
            "manufacturer": [], "rx": False, "ocr_conf": 0.0, "raw_text": ""}
    if not ocr_lines:
        return info
    exp_dates, mfg_dates, other_dates = [], [], []
    texts = [t for t, _ in ocr_lines]
    fields = line_clf.predict([t.lower() for t in texts])
    info["ocr_conf"] = float(np.mean([c for _, c in ocr_lines]))
    info["raw_text"] = "\n".join(texts)
    for (text, conf), field in zip(ocr_lines, fields):
        low = text.lower()
        # rule-based overrides for unambiguous patterns
        if re.search(r"\b(exp|expiry|use before)", low): field = "expiry"
        info["lines"].append({"text": text, "field": field, "conf": round(conf, 2)})
        if field == "expiry":
            d = _parse_date(text)
            if d:
                if re.search(r"\b(exp|use before|use by)", low):     exp_dates.append(d)
                elif re.search(r"\bm\W?[fe]\W?[gd]|manuf|\bmfg", low): mfg_dates.append(d)
                else:                                                other_dates.append(d)
        elif field == "batch":
            m = re.search(r"(?:batch|b\.?\s*no|lot)[\s.:#no]*([a-z0-9\-]{4,})", low)
            info["batch"] = m.group(1).upper() if m else text
        elif field == "directions": info["directions"].append(text)
        elif field == "warning":    info["warnings"].append(text)
        elif field == "storage":    info["storage"].append(text)
        elif field == "manufacturer": info["manufacturer"].append(text)
        elif field == "rx":         info["rx"] = True
        if re.search(r"\brx\b|schedule\s*h|prescription", low): info["rx"] = True
        if field in ("drug_name", "composition") or info["drug"] is None:
            g, s = identify_drug(text)
            if g and s >= info["drug_match"]:
                info["drug"], info["drug_match"] = g, s
        if info["strength_mg"] is None or field in ("drug_name", "composition"):
            m = re.search(r"(\d+(?:\.\d+)?)\s*(mg|mcg|g)\b", low)
            if m:
                v = float(m.group(1)) * {"mg": 1, "mcg": 0.001, "g": 1000}[m.group(2)]
                if field in ("drug_name", "composition") or info["strength_mg"] is None:
                    info["strength_mg"] = v
    # Date reasoning: keyword-tagged dates first; an untagged date is expiry if later than the mfg date
    info["exp"] = max(exp_dates) if exp_dates else None
    info["mfg"] = min(mfg_dates) if mfg_dates else None
    for d in other_dates:
        if info["exp"] is None and (info["mfg"] is None or d > info["mfg"]): info["exp"] = d
        elif info["mfg"] is None and info["exp"] and d < info["exp"]:    info["mfg"] = d
    if info["exp"] and info["mfg"] and info["mfg"] > info["exp"]:
        info["mfg"], info["exp"] = info["exp"], info["mfg"]
    return info

info = extract_info(run_ocr(sample))
{k: v for k, v in info.items() if k not in ("lines", "raw_text")}

## 7. Reasoning: Safety Rules

| # | Rule | Condition | Response |
|---|---|---|---|
| R1 | **Unreadable image** | no text, or mean OCR confidence < 0.40 | Ask for a clearer photo (lighting, focus, flat label) |
| R2 | **Unknown medicine** | no drug matched in the KB | Read out the text, advise checking with a pharmacist |
| R3 | **Expired** | expiry date < today | ❌ Do not use; dispose safely |
| R4 | **Expiring soon** | expires within 90 days | ⚠️ Use soon / check before use |
| R5 | **Prescription-only** | 'Rx' / Schedule H on label or in KB | Use only as prescribed by a doctor |
| R6 | **Dose above maximum** | units × frequency × strength > KB max daily dose | ⚠️ Verify the directions with a pharmacist |
| R7 | **Allergy conflict** | drug belongs to a class in the user's allergy profile | 🚫 Do not take; consult a doctor |
| R8 | **Child safety** | always | Remind to keep out of reach of children |

In [ ]:
USER_PROFILE = {"name": "User", "allergies": ["penicillin"]}   # edit to personalise

def apply_rules(info, profile=USER_PROFILE, today=TODAY):
    alerts, fired = [], []
    if not info["raw_text"] or info["ocr_conf"] < 0.40:
        fired.append("R1")
        return ["📷 I couldn't read this label clearly. Please retake the photo in good light, hold the "
                "camera steady and keep the label flat and close."], fired
    g = info["drug"]
    if g is None:
        fired.append("R2")
        alerts.append("❓ I couldn't identify this medicine in my knowledge base. Please confirm it with a pharmacist.")
    if info["exp"]:
        days = (info["exp"] - today).days
        if days < 0:
            fired.append("R3")
            alerts.append(f"❌ EXPIRED on {info['exp']:%b %Y}. Do NOT use it. Dispose of it safely (pharmacy take-back).")
        elif days <= 90:
            fired.append("R4")
            alerts.append(f"⚠️ Expires soon ({info['exp']:%b %Y}, about {days} days). Check before use.")
    if info["rx"] or (g and DRUG_KB[g]["rx"]):
        fired.append("R5")
        alerts.append("🩺 Prescription medicine: take only as prescribed by your doctor.")
    if g and info["directions"] and info["strength_mg"]:
        units, freq = parse_daily_units(" ".join(info["directions"]))
        if units and freq:
            daily = units * freq * info["strength_mg"]
            info["daily_mg"] = daily
            if daily > DRUG_KB[g]["max_daily_mg"]:
                fired.append("R6")
                alerts.append(f"⚠️ The directions add up to {daily:.0f} mg/day, above the usual adult maximum of "
                              f"{DRUG_KB[g]['max_daily_mg']} mg/day for {g}. Verify with a pharmacist.")
    if g:
        for allergy in profile.get("allergies", []):
            if g in ALLERGY_GROUPS.get(allergy.lower(), []) or allergy.lower() == g:
                fired.append("R7")
                alerts.append(f"🚫 ALLERGY ALERT: {g} belongs to the '{allergy}' group listed in your allergies. Do not take it; consult your doctor.")
    fired.append("R8")
    alerts.append("👶 Keep all medicines out of reach of children.")
    return alerts, fired

print(apply_rules(info))

## 8. The Cognitive Assistant
`MedicineLabelAssistant` ties everything together:
- `.read(image)` → OCR → extraction → rules → spoken and printed **summary**
- `.ask(question)` → answers follow-up questions about the last label (*What is it for? How do I take it? Is it expired? Side effects? Can I take it if I'm allergic to penicillin?*)

In [ ]:
QUESTION_INTENTS = {
    "uses":         ["what is it for", "used for", "use of", "purpose", "treat", "why", "what does it do", "what is this"],
    "directions":   ["how to take", "how do i take", "dose", "dosage", "how many", "how often", "when should", "directions"],
    "expiry":       ["expire", "expiry", "expired", "valid", "still good", "safe to use", "use by", "date"],
    "side_effects": ["side effect", "adverse", "reaction"],
    "warnings":     ["warning", "caution", "careful", "avoid", "precaution", "alcohol", "pregnan", "drive"],
    "storage":      ["store", "storage", "keep", "fridge", "temperature"],
    "allergy":      ["allerg"],
    "identity":     ["name", "which medicine", "what medicine", "generic", "brand", "strength", "mg"],
    "manufacturer": ["manufactur", "company", "made by", "batch", "lot"],
}

class MedicineLabelAssistant:
    def __init__(self, profile=USER_PROFILE):
        self.profile, self.info, self.alerts = profile, None, []

    def read(self, image, show=True, speak=False):
        self.info = extract_info(run_ocr(image))
        self.alerts, self.fired = apply_rules(self.info, self.profile)
        summary = self.summary()
        if show:
            plt.figure(figsize=(7, 5)); plt.imshow(image); plt.axis("off"); plt.show()
            print(summary)
        if speak:
            self.speak(summary)
        return summary

    def summary(self):
        i = self.info
        if "R1" in self.fired:
            return self.alerts[0]
        parts = []
        if i["drug"]:
            k = DRUG_KB[i["drug"]]
            strength = f" {i['strength_mg']:g} mg" if i["strength_mg"] else ""
            parts.append(f"💊 This is **{i['drug'].capitalize()}{strength}** ({k['class']}).")
            parts.append(f"   Used for: {k['uses']}.")
        if i["directions"]:
            parts.append(f"   How to take (label): {' '.join(i['directions'])}")
        if i["exp"]:
            parts.append(f"   Expiry: {i['exp']:%b %Y}" + (f" | Mfg: {i['mfg']:%b %Y}" if i["mfg"] else ""))
        if i["batch"]:
            parts.append(f"   Batch: {i['batch']}")
        parts.append("")
        parts += self.alerts
        return "\n".join(parts)

    def _intent(self, q):
        q = q.lower()
        scores = {k: sum(p in q for p in pats) for k, pats in QUESTION_INTENTS.items()}
        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else None

    def ask(self, question):
        if self.info is None or not self.info["raw_text"]:
            return "Please show me a medicine label first."
        i, g = self.info, self.info["drug"]
        k = DRUG_KB.get(g, {})
        intent = self._intent(question)
        if intent is None:
            return "I can answer: what it's for, how to take it, expiry, side effects, warnings, storage, allergies or the manufacturer."
        if intent in ("uses", "side_effects", "allergy") and not g:
            return "I couldn't identify this medicine, so please ask a pharmacist about that."
        if intent == "uses":
            return f"{g.capitalize()} is used for {k['uses']}."
        if intent == "directions":
            label = " ".join(i["directions"]) or "I couldn't find directions on the label."
            extra = f" That's about {i['daily_mg']:.0f} mg per day." if i.get("daily_mg") else ""
            return f"The label says: {label}{extra} Always follow your doctor's instructions."
        if intent == "expiry":
            if not i["exp"]:
                return "I couldn't read an expiry date. Please check the label manually."
            days = (i["exp"] - TODAY).days
            return (f"It expired in {i['exp']:%B %Y}. Please don't use it." if days < 0 else
                    f"It's valid until {i['exp']:%B %Y} ({days} days from today).")
        if intent == "side_effects":
            return f"Common side effects of {g}: {k['side_effects']}. Contact a doctor if anything severe happens."
        if intent == "warnings":
            label = " ".join(i["warnings"])
            return (f"Label warnings: {label} " if label else "") + (f"General warnings: {k['warnings']}." if k else "")
        if intent == "storage":
            return " ".join(i["storage"]) or "Store in a cool, dry place away from sunlight unless the label says otherwise."
        if intent == "allergy":
            m = re.search(r"allergic to (\w+)", question.lower())
            allergy = m.group(1) if m else None
            if allergy and (g in ALLERGY_GROUPS.get(allergy, []) or allergy == g):
                return f"🚫 No. {g.capitalize()} is in the {allergy} group. Do not take it if you're allergic; consult your doctor."
            if allergy:
                return f"{g.capitalize()} isn't in the {allergy} group in my knowledge base, but confirm with a pharmacist before taking it."
            return "Tell me what you're allergic to, e.g. 'I'm allergic to penicillin, can I take this?'"
        if intent == "identity":
            return f"It's {g or 'an unidentified medicine'}" + (f", {i['strength_mg']:g} mg" if i["strength_mg"] else "") + "."
        if intent == "manufacturer":
            return " ".join(i["manufacturer"]) + (f" Batch: {i['batch']}." if i["batch"] else "")

    @staticmethod
    def speak(text):
        """Text-to-speech for visually-impaired users (needs internet, as in Colab)."""
        try:
            from gtts import gTTS
            from IPython.display import Audio, display
            gTTS(re.sub(r"[*❌⚠️🩺👶🚫💊❓📷]", "", text)).save("speech.mp3")
            display(Audio("speech.mp3", autoplay=False))
        except Exception as e:
            print("(speech unavailable:", e, ")")

assistant = MedicineLabelAssistant()

## 9. Evaluation: End-to-End on Unseen Label Images
40 **new** label photos (not used for training) go through the full pipeline (image → OCR → extraction → rules), at two difficulty levels:
- **normal**: slight tilt, mild blur and noise (a decent phone photo)
- **hard**: strong tilt, heavy blur, low resolution, dim low-contrast lighting and JPEG artefacts

Extracted fields are compared with the ground truth. The line classifier is also scored on the **actual OCR output lines** (each aligned to its most similar ground-truth line).

In [ ]:
def evaluate(n=40, hard=False, seed=0):
    random.seed(seed); np.random.seed(seed)
    recs = [make_label_record() for _ in range(n)]
    rows, line_true, line_pred = [], [], []
    for r in recs:
        inf = extract_info(run_ocr(render_label(r, hard=hard)))
        _, fired = apply_rules(inf)
        for item in inf["lines"]:
            sims = [difflib.SequenceMatcher(None, item["text"].lower(), t.lower()).ratio() for _, t in r["lines"]]
            if max(sims) >= 0.6:
                line_true.append(r["lines"][int(np.argmax(sims))][0]); line_pred.append(item["field"])
        exp_true = date.fromisoformat(r["exp"])
        rows.append({"drug_true": r["generic"], "drug_pred": inf["drug"],
                     "strength_ok": inf["strength_mg"] == r["strength_mg"], "exp_ok": inf["exp"] == exp_true,
                     "expired_true": exp_true < TODAY, "expired_pred": "R3" in fired,
                     "rx_ok": inf["rx"] == r["rx"], "unreadable": "R1" in fired, "ocr_conf": inf["ocr_conf"]})
    ev = pd.DataFrame(rows)
    m = {"Line-field classification (OCR lines)": accuracy_score(line_true, line_pred) if line_true else 0,
         "Drug identification": (ev["drug_true"] == ev["drug_pred"]).mean(),
         "Strength extraction": ev["strength_ok"].mean(),
         "Expiry-date extraction": ev["exp_ok"].mean(),
         "Expired / valid decision": (ev["expired_true"] == ev["expired_pred"]).mean(),
         "Prescription status": ev["rx_ok"].mean(),
         "Flagged unreadable (R1)": ev["unreadable"].mean(),
         "Mean OCR confidence": ev["ocr_conf"].mean()}
    return m, ev, (line_true, line_pred)

results, details = {}, {}
for level in ["normal", "hard"]:
    results[level], ev, lines = evaluate(hard=(level == "hard"), seed=123)
    details[level] = (ev, lines)
res = pd.DataFrame(results).round(3)
print(f"OCR engine: {OCR_ENGINE}")
res

In [ ]:
res.iloc[:6].plot.barh(figsize=(9, 4), color=["#2a9d8f", "#e76f51"])
plt.xlim(0, 1.05); plt.title("End-to-end accuracy on 40 unseen label photos"); plt.tight_layout(); plt.show()

ev_h, (lt, lp) = details["hard"]
print(f"Line classifier on {len(lt)} OCR lines from HARD images:")
print(classification_report(lt, lp, digits=3, zero_division=0))
print("Failures on hard images:")
ev_h[(ev_h["drug_true"] != ev_h["drug_pred"]) | ~ev_h["exp_ok"]].head(8)

## 10. Demonstration on Synthetic Labels: Summary, Rules and Q&A

In [ ]:
# Pick an interesting case: an Rx antibiotic (amoxicillin) for a user with a penicillin allergy
random.seed(7)
demo = make_label_record()
while demo["generic"] != "amoxicillin":
    demo = make_label_record()
assistant.read(render_label(demo))
for q in ["What is this medicine for?", "How do I take it?", "Is it expired?",
          "What are the side effects?", "I'm allergic to penicillin, can I take this?", "How should I store it?"]:
    print(f"\n🧑 {q}\n🤖 {assistant.ask(q)}")

In [ ]:
# A label whose directions exceed the maximum daily dose (rule R6), and an expired one (R3)
bad = make_label_record()
bad.update(generic="paracetamol", brand="DOLO", strength_mg=650)
bad["lines"] = [("drug_name", "DOLO 650"), ("drug_name", "Paracetamol Tablets IP 650 mg"),
                ("directions", "Dosage: Take 2 tablets every 4 hours."), ("warning", "Keep out of reach of children."),
                ("batch", "Batch No. DL22041"), ("expiry", "MFG. 03/2023"), ("expiry", "EXP. 02/2025"),
                ("manufacturer", "Mfd. by: Sunrise Pharma Ltd.")]
assistant.read(render_label(bad))

## 11. Real Medicine Packaging Images (from the Internet)
Downloads real photos of over-the-counter medicine packs from Hugging Face ([`mennox/Italian-OTC-medicines`](https://huggingface.co/datasets/mennox/Italian-OTC-medicines)) and runs the assistant on them. These labels are in Italian, so expect drug matching on generic names like *paracetamolo → paracetamol* or *ibuprofene → ibuprofen*. Many strings won't be in the KB, which exercises rule R2.

In [ ]:
try:
    from datasets import load_dataset
    real = load_dataset("mennox/Italian-OTC-medicines", split="train")
    print("Loaded", len(real), "real images")
    for idx in [0, 5, 12, 40]:
        img = real[idx]["image"].convert("RGB")
        print(f"\n━━━━━━━━ Real image #{idx} ━━━━━━━━")
        assistant.read(img)
        print("\nOCR text:\n" + assistant.info["raw_text"][:400])
except Exception as e:
    print("Could not download the real-image dataset:", e, "\n→ Use the upload cell below instead.")

## 12. Try It Yourself: Upload a Photo of a Medicine Bottle
Run the cell, upload one or more photos (JPG/PNG), then ask questions in the box that appears. Set `speak=True` to hear the summary read aloud.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        print(f"\n━━━━━━━━ {name} ━━━━━━━━")
        assistant.read(Image.open(name).convert("RGB"), speak=True)
except Exception as e:
    print("Upload works in Google Colab:", e)

In [ ]:
# Interactive Q&A about the most recently read label (type 'quit' to stop)
def chat():
    print("Ask about the last label (e.g. 'What is it for?', 'Is it expired?'). Type 'quit' to exit.")
    while True:
        q = input("🧑 ")
        if q.strip().lower() in {"quit", "exit", "q"}:
            break
        print("🤖", assistant.ask(q))
chat()

## Conclusion
- **Perception:** OpenCV pre-processing (denoise, CLAHE) + EasyOCR deep-learning text detection and recognition.
- **Comprehension:** a trained character-n-gram TF-IDF + Logistic Regression **line-field classifier** (9 classes, trained with simulated OCR noise), plus fuzzy drug-name matching and regex extraction of strength, dates, batch, dose and frequency.
- **Reasoning:** drug knowledge base (12 drugs, about 40 names) and 8 safety rules: unreadable image, unknown drug, expired, expiring soon, prescription-only, dose above maximum, allergy conflict, child safety.
- **Interaction:** a natural-language summary, follow-up Q&A and text-to-speech for accessibility.
- **Limitations:** curved bottle labels, glare and very small fonts reduce OCR accuracy; the KB is small; the synthetic labels are cleaner than real photos. Improvements: label region detection (e.g. a LandingLens / YOLO detector) before OCR, a larger drug database (e.g. CDSCO / openFDA), and a fine-tuned OCR model on real pharmacy labels.